# Employee Attrition Prediction

**Machine Learning Minor Project**  
**Author:** Namami Sharma  
**College:** Samrat Ashoka Technological Institute, Vidisha

### Objective
Predict whether an employee is likely to leave the company (`Yes`) or stay (`No`) using employee-related information.

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')

## 2. Load the Dataset

Keep `Employee_Attrition_Dataset.csv` in the same folder as this notebook.

In [ ]:
df = pd.read_csv('Employee_Attrition_Dataset.csv')
df.head()

In [ ]:
print('Dataset shape:', df.shape)
print('\nColumn names:')
print(df.columns.tolist())


## 3. Understand the Dataset

In [ ]:
df.info()


In [ ]:
print('Missing values in each column:')
print(df.isnull().sum())

print('\nDuplicate rows:', df.duplicated().sum())


In [ ]:
df.describe(include='all').T

## 4. Exploratory Data Analysis (EDA)

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='Attrition')
plt.title('Employee Attrition Distribution')
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='OverTime', hue='Attrition')
plt.title('Overtime vs Attrition')
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='Attrition', y='MonthlyIncome')
plt.title('Monthly Income vs Attrition')
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='JobSatisfaction', hue='Attrition')
plt.title('Job Satisfaction vs Attrition')
plt.show()


## 5. Prepare Data for Machine Learning

The target column is `Attrition`. Categorical columns are converted into numerical form using One-Hot Encoding.

In [ ]:
X = df.drop('Attrition', axis=1)
y = df['Attrition'].map({'No': 0, 'Yes': 1})

categorical_features = X.select_dtypes(include=['object']).columns.tolist()
numeric_features = X.select_dtypes(exclude=['object']).columns.tolist()

print('Categorical columns:', categorical_features)
print('Numeric columns:', numeric_features)

In [ ]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print('Training samples:', X_train.shape[0])
print('Testing samples:', X_test.shape[0])

## 6. Train Machine Learning Models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=6, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42)
}

results = []
trained_models = {}

for name, model in models.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1 Score': f1_score(y_test, y_pred, zero_division=0)
    })
    trained_models[name] = pipeline

results_df = pd.DataFrame(results).sort_values('F1 Score', ascending=False)
results_df

## 7. Evaluate the Models

In [ ]:
for name, pipeline in trained_models.items():
    y_pred = pipeline.predict(X_test)
    print('=' * 60)
    print(name)
    print('=' * 60)
    print(classification_report(y_test, y_pred, target_names=['Stay', 'Leave'], zero_division=0))


In [ ]:
best_model_name = results_df.iloc[0]['Model']
best_model = trained_models[best_model_name]
best_predictions = best_model.predict(X_test)

cm = confusion_matrix(y_test, best_predictions)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Stay', 'Leave'],
            yticklabels=['Stay', 'Leave'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Confusion Matrix - {best_model_name}')
plt.show()


## 8. Make a Prediction for a New Employee

Change the values below to test the model with a new employee.

In [ ]:
new_employee = pd.DataFrame([{
    'Age': 25,
    'Gender': 'Female',
    'Department': 'Sales',
    'JobRole': 'Sales Executive',
    'MaritalStatus': 'Single',
    'MonthlyIncome': 3500,
    'YearsAtCompany': 2,
    'JobSatisfaction': 2,
    'WorkLifeBalance': 2,
    'JobInvolvement': 2,
    'EnvironmentSatisfaction': 2,
    'OverTime': 'Yes',
    'BusinessTravel': 'Travel_Frequently',
    'DistanceFromHome': 15,
    'NumCompaniesWorked': 3,
    'TrainingTimesLastYear': 2,
    'StockOptionLevel': 0,
    'PerformanceRating': 3,
    'Education': 3
}])

prediction = best_model.predict(new_employee)[0]
probability = best_model.predict_proba(new_employee)[0][1]

print('Selected model:', best_model_name)
print('Prediction:', 'Employee may leave' if prediction == 1 else 'Employee may stay')
print(f'Estimated probability of attrition: {probability:.2%}')

## 9. Conclusion

This project demonstrates a complete Machine Learning workflow for employee attrition prediction: data loading, exploration, preprocessing, model training, evaluation and prediction.

**Note:** This dataset is synthetic and is intended for educational/project demonstration purposes. Model predictions should not be used as the sole basis for real employment decisions.